# Session 3 · Functions, modules and files

Work top to bottom and run each cell with **Shift+Enter**.

Cells under a **Your turn** heading contain `...` placeholders. Replace them, run the cell, then run the check cell beneath it. Checks print `PASS` or `NOT YET` and never stop the notebook.

Solutions are in the matching `_solutions` notebook. Open it only after a genuine attempt.

**Before you start:** copy `matches_2025_synthetic.csv` into a `data/` folder at the top of your project (the Session 3 guide shows the command). This notebook lives in `sessions/`, so it reads the file from `../data/matches_2025_synthetic.csv`.

In [24]:
def check(label, get, expected):
    """Print PASS or NOT YET. `get` is a lambda, so missing names or errors never stop the notebook."""
    try:
        got = get()
    except Exception as e:
        print(f"NOT YET  {label}: {type(e).__name__}: {e}")
        return
    try:
        ok = got.equals(expected) if hasattr(got, "equals") else bool(got == expected)
        if isinstance(expected, float) and isinstance(got, (int, float)):
            ok = abs(got - expected) < 1e-6
    except Exception:
        ok = False
    print(f"PASS     {label}" if ok else f"NOT YET  {label}: expected {expected!r}, got {got!r}")

## 1. Functions

| R | Python |
| --- | --- |
| `f <- function(x, y = 2) { ... }` | `def f(x, y=2):` followed by an indented body |
| The last expression is returned automatically | You **must** write `return`; without it the function returns `None` |
| `f(y = 3, x = 1)` | `f(y=3, x=1)`, keyword arguments work the same way |
| `?f` or roxygen comments | `help(f)` reads the **docstring**, the string on the first line of the body |

In [25]:
def total_score(goals, behinds):
    """Return an AFL score: 6 points per goal, 1 per behind."""
    return goals * 6 + behinds


print(total_score(15, 12))
print(total_score(behinds=12, goals=15))   # keyword arguments, any order
help(total_score)                          # prints the docstring

102
102
Help on function total_score in module __main__:

total_score(goals, behinds)
    Return an AFL score: 6 points per goal, 1 per behind.



The R habit that bites hardest: forgetting `return`. Python runs the line, throws the answer away, and quietly hands back `None`.

In [26]:
def total_score_forgot(goals, behinds):
    goals * 6 + behinds          # R would return this. Python does not.


print(total_score_forgot(15, 12))   # None

None


### Your turn 1.1

Write `margin(home_points, away_points, absolute=False)`. It returns home minus away. When `absolute=True` it returns the size of the margin regardless of who won (`abs()` is the same as in R). Give it a one-line docstring.

In [27]:
def margin(home_points, away_points, absolute=False):
    """This function returns the margin of a game. If absolute is True, it returns the absolute value of the margin."""
    if absolute:
        return abs(home_points - away_points)
    return home_points - away_points

In [28]:
check("1.1 home win", lambda: margin(102, 62), 40)
check("1.1 away win", lambda: margin(62, 102), -40)
check("1.1 absolute", lambda: margin(62, 102, absolute=True), 40)

PASS     1.1 home win
PASS     1.1 away win
PASS     1.1 absolute


### Your turn 1.2

Write `result(home_points, away_points)` that returns the string `"home"`, `"away"` or `"draw"`. Use `if`, `elif` and `else`.

In [29]:
def result(home_points, away_points):
    """Return 'home', 'away' or 'draw'."""
    if home_points > away_points:
        return "home"
    elif home_points < away_points:
        return "away"
    else:
        return "draw"

In [30]:
check("1.2 home", lambda: result(102, 62), "home")
check("1.2 away", lambda: result(62, 102), "away")
check("1.2 draw", lambda: result(80, 80), "draw")

PASS     1.2 home
PASS     1.2 away
PASS     1.2 draw


### Your turn 1.3

Python functions can return several values at once as a **tuple**, which you unpack on the way out: `g, b, t = parse_score("15.12")`. This replaces R's habit of returning a list.

Write `parse_score(text)` that turns `"15.12"` into `(15, 12, 102)`: goals, behinds and total, all as integers. Hint: `text.split(".")` gives `["15", "12"]`.

In [31]:
def parse_score(text):
    """Turn 'goals.behinds' into (goals, behinds, total)."""
    goals, behinds = map(int, text.split("."))
    return goals, behinds, goals * 6 + behinds

In [32]:
check("1.3 Geelong", lambda: parse_score("15.12"), (15, 12, 102))
check("1.3 Sydney", lambda: parse_score("9.8"), (9, 8, 62))

PASS     1.3 Geelong
PASS     1.3 Sydney


### One trap worth knowing now: mutable defaults

A default value is created **once**, when the function is defined, not each time it is called. With a list as the default, every call shares the same list. Use `None` and create the list inside.

In [33]:
def add_team(team, teams=[]):          # don't do this
    teams.append(team)
    return teams


print(add_team("Geelong"))
print(add_team("Sydney"))              # ['Geelong', 'Sydney']: the list survived between calls


def add_team_safe(team, teams=None):   # do this instead
    if teams is None:
        teams = []
    teams.append(team)
    return teams


print(add_team_safe("Geelong"))
print(add_team_safe("Sydney"))

['Geelong']
['Geelong', 'Sydney']
['Geelong']
['Sydney']


## 2. Modules

A module is just a `.py` file. `import footy` is like `source("footy.R")`, with one big difference: the functions stay inside a namespace, so you call `footy.total_score()`, much like `pkg::fn()` in R.

Normally you would create `footy.py` in VS Code. Here the `%%writefile` line at the top of a cell writes the rest of that cell to a file, so everything stays in one notebook. The file lands in the same folder as this notebook.

In [34]:
%%writefile footy.py
"""Helpers for football scores."""


def total_score(goals, behinds):
    """Return the score from goals and behinds."""
    return goals * 6 + behinds

Overwriting footy.py


In [35]:
import importlib

import footy

importlib.reload(footy)   # re-reads the file after you edit it; otherwise Python keeps the old version
print(footy.total_score(15, 12))

from footy import total_score as ts   # import one function under a short name
print(ts(9, 8))

102
62


### Your turn 2.1

Rewrite `footy.py` so it also contains your `margin()` and `result()` functions from section 1. Paste them in under `total_score()`, run the cell, then run the check cell, which reloads the module first.

Open `footy.py` in VS Code afterwards: it is an ordinary file you could import from any script in this folder.

In [36]:
%%writefile footy.py
"""Helpers for football scores."""


def total_score(goals, behinds):
    """Return the score from goals and behinds."""
    return goals * 6 + behinds


def margin(home_points, away_points, absolute=False):
    """This function returns the margin of a game. If absolute is True, it returns the absolute value of the margin."""
    if absolute:
        return abs(home_points - away_points)
    return home_points - away_points


def result(home_points, away_points):
    """Return 'home', 'away' or 'draw'."""
    if home_points > away_points:
        return "home"
    elif home_points < away_points:
        return "away"
    else:
        return "draw"

Overwriting footy.py


In [37]:
importlib.reload(footy)
check("2.1 margin", lambda: footy.margin(102, 62), 40)
check("2.1 result", lambda: footy.result(80, 80), "draw")

PASS     2.1 margin
PASS     2.1 result


**Why `main.py` ends with `if __name__ == "__main__":`.** When Python imports a file, it runs every top-level line in it. That guard means "only run this block when the file is executed directly, not when it is imported". Keep functions at the top level and put script-style code under the guard.

## 3. Reading a CSV with pandas

`pd.read_csv()` is the equivalent of `readr::read_csv()`. The data is a **synthetic** 2025 season, invented for practice, so do not quote these results anywhere.

`parse_dates=["date"]` converts the date column on the way in, like `col_types` in readr.

In [38]:
import pandas as pd

matches = pd.read_csv("../data/matches_2025_synthetic.csv", parse_dates=["date"])
print(matches.shape)   # dim(matches): (rows, columns)
matches.head()         # head(matches)

(207, 11)


,match_id,season,round,date,home_team,away_team,home_goals,home_behinds,away_goals,away_behinds,crowd
0,1,2025,1,2025-03-16,Collingwood,Carlton,12,12,19,11,14893.0
1,2,2025,1,2025-03-16,Hawthorn,Brisbane,14,9,17,10,12927.0
2,3,2025,1,2025-03-15,North Melbourne,Richmond,16,11,10,9,21940.0
3,4,2025,1,2025-03-15,Essendon,West Coast,10,12,13,4,64499.0
4,5,2025,1,2025-03-15,GWS,Fremantle,4,10,14,9,33167.0


In [39]:
matches.info()          # like dplyr::glimpse(): columns, types and non-missing counts

<class 'pandas.DataFrame'>
RangeIndex: 207 entries, 0 to 206
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   match_id      207 non-null    int64         
 1   season        207 non-null    int64         
 2   round         207 non-null    int64         
 3   date          207 non-null    datetime64[us]
 4   home_team     207 non-null    str           
 5   away_team     207 non-null    str           
 6   home_goals    207 non-null    int64         
 7   home_behinds  207 non-null    int64         
 8   away_goals    207 non-null    int64         
 9   away_behinds  207 non-null    int64         
 10  crowd         200 non-null    float64       
dtypes: datetime64[us](1), float64(1), int64(7), str(2)
memory usage: 17.9 KB


In [40]:
matches.describe()      # like summary() for the numeric columns

,match_id,season,round,date,home_goals,home_behinds,away_goals,away_behinds,crowd
count,207.000000,207.0,207.00000,207,207.000000,207.000000,207.000000,207.000000,200.000000
mean,104.000000,2025.0,12.00000,2025-05-30 13:54:46.956521,12.526570,10.415459,11.512077,10.661836,35323.955000
min,1.000000,2025.0,1.00000,2025-03-13 00:00:00,4.000000,3.000000,2.000000,4.000000,7705.000000
25%,52.500000,2025.0,6.00000,2025-04-19 00:00:00,10.000000,8.000000,9.000000,9.000000,24777.500000
50%,104.000000,2025.0,12.00000,2025-05-31 00:00:00,12.000000,10.000000,11.000000,11.000000,32070.000000
75%,155.500000,2025.0,18.00000,2025-07-10 00:00:00,15.000000,12.000000,14.000000,13.000000,42362.000000
max,207.000000,2025.0,23.00000,2025-08-16 00:00:00,24.000000,21.000000,21.000000,20.000000,95000.000000
std,59.899917,0.0,6.64933,NaN,3.970525,3.218779,3.598624,3.069050,15715.764379


### Your turn 3.1

Using `matches`, set:

- `n_matches`: the number of rows (`len()` works on a DataFrame)
- `n_teams`: the number of distinct home teams (`.nunique()` on a column)
- `mean_crowd`: the mean crowd, rounded to a whole number with `round()`. Missing crowds are skipped automatically, like `na.rm = TRUE`.

Select a single column with `matches["home_team"]`, much like `matches$home_team`.

In [41]:
n_matches = len(matches)
n_teams = matches["home_team"].nunique() # the number of distinct home teams (.nunique() on a column)
mean_crowd = round(matches["crowd"].mean())

In [42]:
check("3.1 n_matches", lambda: n_matches, 207)
check("3.1 n_teams", lambda: n_teams, 18)
check("3.1 mean_crowd", lambda: mean_crowd, 35324)

PASS     3.1 n_matches
PASS     3.1 n_teams
PASS     3.1 mean_crowd


## Wrap-up

1. Run All and confirm every check prints `PASS`.
2. Commit: `git add sessions/ data/` then `git commit -m "Session 3: functions, modules, first CSV"` then `git push`.
3. Move on to the Exercism exercises and the Milestone 1 decision in the Session 3 guide.